In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import librosa

processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [ ]:
num_map = {
    "शून्य": 0, "एक": 1, "दो": 2, "तीन": 3, "चार": 4,
    "पांच": 5, "छह": 6, "सात": 7, "आठ": 8, "नौ": 9,
    "दस": 10, "ग्यारह": 11, "बारह": 12, "तेरह": 13,
    "चौदह": 14, "पंद्रह": 15, "सोलह": 16, "सत्रह": 17,
    "अठारह": 18, "उन्नीस": 19, "बीस": 20,
    "तीस": 30, "चालीस": 40, "पचास": 50,
    "साठ": 60, "सत्तर": 70, "अस्सी": 80, "नब्बे": 90,
    "चौवन": 54,  # add missing ones like this
    "सौ": 100, "हजार": 1000
}

In [ ]:
def hindi_to_number(words):
    total = 0
    current = 0

    for word in words:
        val = num_map.get(word, None)
        if val is None:
            continue

        if val == 100:
            current = max(1, current) * 100
        elif val == 1000:
            current = max(1, current) * 1000
            total += current
            current = 0
        else:
            current += val

    total += current
    return total

In [ ]:
import re

def normalize_numbers(text):
    words = text.split()
    result = []
    buffer = []

    for word in words:
        if word in num_map:
            buffer.append(word)
        else:
            if buffer:
                result.append(str(hindi_to_number(buffer)))
                buffer = []
            result.append(word)

    if buffer:
        result.append(str(hindi_to_number(buffer)))

    return " ".join(result)

In [ ]:
def is_safe_to_convert(text, word):
    # skip hyphen cases
    if "-" in text:
        return False
    return True

In [ ]:
english_words = [
    "इंटरव्यू", "कंप्यूटर", "प्रॉब्लम", "जॉब",
    "मीटिंग", "मैनेजमेंट", "प्रोजेक्ट", "डेटा",
    "कोड", "सिस्टम", "नेटवर्क"
]

In [ ]:
import re

def is_english_word(word):
    # Rule 1: dictionary
    if word in english_words:
        return True

    # Rule 2: roman script
    if re.search(r"[a-zA-Z]", word):
        return True

    # Rule 3: phonetic endings
    if word.endswith(("शन", "मेंट", "टिंग")):
        return True

    return False

In [ ]:
def tag_english_words(text):
    words = text.split()
    result = []

    for word in words:
        if is_english_word(word):
            result.append(f"[EN]{word}[/EN]")
        else:
            result.append(word)

    return " ".join(result)

In [ ]:
def remove_repetition(text):
    words = text.split()
    result = []

    for word in words:
        if len(result) < 2 or not (word == result[-1] == result[-2]):
            result.append(word)

    return " ".join(result)

In [ ]:
def full_pipeline(text):
    text = remove_repetition(text)
    text = normalize_numbers(text)
    text = tag_english_words(text)
    return text

#### TESTING PIPELINE

In [ ]:
texts = [
    "मेरे पास दो किताबें हैं",
    "यह प्रॉब्लम solve नहीं हो रहा",
    "उसने तीन सौ चौवन रुपये दिए",
    "दो-चार बातें हुई",
    "मेरा इंटरव्यू अच्छा गया"
]

for t in texts:
    print("Input :", t)
    print("Output:", full_pipeline(t))
    print("-"*40)

Input : मेरे पास दो किताबें हैं
Output: मेरे पास 2 किताबें हैं
----------------------------------------
Input : यह प्रॉब्लम solve नहीं हो रहा
Output: यह [EN]प्रॉब्लम[/EN] [EN]solve[/EN] नहीं हो रहा
----------------------------------------
Input : उसने तीन सौ चौवन रुपये दिए
Output: उसने 354 रुपये दिए
----------------------------------------
Input : दो-चार बातें हुई
Output: दो-चार बातें हुई
----------------------------------------
Input : मेरा इंटरव्यू अच्छा गया
Output: मेरा [EN]इंटरव्यू[/EN] अच्छा गया
----------------------------------------


In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/josh_talks_asr/segments_metadata.csv")
df.head()


,recording_id,user_id,rec_url_gcp,start,end,speaker_id,text,duration_seg,text_normalized,audio_path
0,825780,245746,https://storage.googleapis.com/upload_goai/967...,0.11,14.42,245746,अब काफी अच्छा होता है क्योंकि उनकी जनसंख्या बह...,14.31,अब काफी अच्छा होता है क्योंकि उनकी जनसंख्या बह...,/content/audio_segments/825780_0.wav
1,825780,245746,https://storage.googleapis.com/upload_goai/967...,14.42,29.03,245746,अनुभव करके कुछ लिखना था तो वह तो बिना देखिए नह...,14.61,अनुभव करके कुछ लिखना था तो वह तो बिना देखिए नह...,/content/audio_segments/825780_1.wav
2,825780,245746,https://storage.googleapis.com/upload_goai/967...,29.03,41.84,245746,जंगल का सफर होता है जब हम रहने के लिए गए थे ना...,12.81,जंगल का सफर होता है जब हम रहने के लिए गए थे ना...,/content/audio_segments/825780_2.wav
3,825780,245746,https://storage.googleapis.com/upload_goai/967...,42.47,50.57,245746,पहली बारी था क्योंकि चलना नहीं आता न वहाँ का ज...,8.10,पहली बारी था क्योंकि चलना नहीं आता न वहाँ का ज...,/content/audio_segments/825780_3.wav
4,825780,245746,https://storage.googleapis.com/upload_goai/967...,52.70,66.83,245746,हां तो फिर वहां जो दिन भर तो दिन में तो खोजने ...,14.13,हां तो फिर वहां जो दिन भर तो दिन में तो खोजने ...,/content/audio_segments/825780_4.wav


In [ ]:
df.columns

Index(['recording_id', 'user_id', 'rec_url_gcp', 'start', 'end', 'speaker_id',
       'text', 'duration_seg', 'text_normalized', 'audio_path'],
      dtype='object')

In [ ]:
import requests
import tempfile
import librosa

for i in range(3):
    row = df.iloc[i]

    url = row["rec_url_gcp"]
    reference = row["text"]

    try:
        # download audio
        response = requests.get(url)
        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
        tmp.write(response.content)

        # load audio
        audio, sr = librosa.load(tmp.name, sr=16000)

        # ASR
        inputs = processor(audio, sampling_rate=sr, return_tensors="pt")
        predicted_ids = model.generate(inputs.input_features)

        transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

        print("Reference:", reference)
        print("Raw ASR:", transcription)
        print("Cleaned :", full_pipeline(transcription))
        print("-"*50)

    except Exception as e:
        print("Error loading audio:", e)

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.log

Reference: अब काफी अच्छा होता है क्योंकि उनकी जनसंख्या बहुत कम दी जा रही है तो हमें उनको देखना था तो एक देखना था मतलब वो तो देखना था लेकिन हमारा प्रोजेक्ट भी था कि जो जन जाती पाई जाती है उधर कि उधर की एरिया में उसके बारे में देखना अब
Raw ASR:  अगर अदर अदर की अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अ
Cleaned : अगर अदर अदर की अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अ
--------------------------------